In [ ]:
!pip install -q pymupdf faiss-cpu sentence-transformers langchain-text-splitters

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 69.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 62.3 MB/s eta 0:00:00


In [ ]:
import re
import fitz
import numpy as np
from langchain_text_splitters import RecursiveCharacterTextSplitter
import random

def preprocess_text(text):
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'[^a-zA-ZąćęłńóśźżĄĆĘŁŃÓŚŹŻ0-9\s\.\,\-\:\(\)\;\!]', '', text)
    text = re.sub(r'-\n\s*', '', text)
    return text.strip()

def find_column_split(text_blocks, page_width):
    x0_values = sorted(set(round(b[0]) for b in text_blocks))
    if len(x0_values) < 2:
        return page_width / 2

    mid_low  = page_width * 0.25
    mid_high = page_width * 0.75
    min_gap  = page_width * 0.05

    best_gap, best_split = 0, None
    for i in range(len(x0_values) - 1):
        gap_mid  = (x0_values[i] + x0_values[i + 1]) / 2
        gap_size =  x0_values[i + 1] - x0_values[i]
        if mid_low <= gap_mid <= mid_high and gap_size > min_gap and gap_size > best_gap:
            best_gap, best_split = gap_size, gap_mid

    return best_split if best_split else page_width / 2

def extract_page_smart_columns(page, col_split_threshold=0.55):
    page_width  = page.rect.width
    blocks      = page.get_text("blocks")

    text_blocks = [b for b in blocks if b[6] == 0 and b[4].strip()]
    if not text_blocks:
        return ""

    split = find_column_split(text_blocks, page_width)

    full_width  = [b for b in text_blocks if (b[2] - b[0]) >  page_width * col_split_threshold]
    col_blocks  = [b for b in text_blocks if (b[2] - b[0]) <= page_width * col_split_threshold]

    left  = sorted([b for b in col_blocks if (b[0] + b[2]) / 2 <  split], key=lambda b: b[1])
    right = sorted([b for b in col_blocks if (b[0] + b[2]) / 2 >= split], key=lambda b: b[1])
    full_width.sort(key=lambda b: b[1])

    ordered, li, ri = [], 0, 0
    for fb in full_width:
        fy = fb[1]
        while li < len(left)  and left[li][1]  < fy: ordered.append(left[li][4]);  li += 1
        while ri < len(right) and right[ri][1] < fy: ordered.append(right[ri][4]); ri += 1
        ordered.append(fb[4])
    ordered.extend(b[4] for b in left[li:])
    ordered.extend(b[4] for b in right[ri:])

    return "\n".join(ordered)

def extract_and_clean(pdf_paths):
    combined_text = ""
    for path in pdf_paths:
        doc = fitz.open(path)
        for page in doc:
            combined_text += extract_page_smart_columns(page) + "\n\n"
        doc.close()
    return preprocess_text(combined_text)

def debug_page(pdf_path, page_num):
    doc   = fitz.open(pdf_path)
    page  = doc[page_num]
    pw    = page.rect.width
    blks  = [b for b in page.get_text("blocks") if b[6] == 0 and b[4].strip()]
    split = find_column_split(blks, pw)
    print(f"Page {page_num} | width={pw:.0f} | detected split={split:.0f}")
    print(f"{'x0':>7} {'x1':>7} {'y0':>7} {'col':<6}  preview")
    print("-" * 70)
    for b in sorted(blks, key=lambda x: x[1]):
        x0, y0, x1 = b[0], b[1], b[2]
        if   (x1 - x0) > pw * 0.55:    col = "FULL"
        elif (x0 + x1) / 2 < split:    col = "LEFT"
        else:                           col = "RIGHT"
        print(f"{x0:>7.1f} {x1:>7.1f} {y0:>7.1f} {col:<6}  {b[4][:60].replace(chr(10), '↵')}")
    doc.close()

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", ". ", " ", ""],
    length_function=len,
)

player_text   = extract_and_clean(['/content/player.pdf'])
player_chunks = text_splitter.split_text(player_text)

master_text   = extract_and_clean(['/content/master.pdf', '/content/monsters.pdf'])
master_chunks = text_splitter.split_text(master_text)

def preview_random_chunks(chunks, name, num_samples=1):
    print(f"\n{'='*20} LOSOWY PODGLĄD: {name.upper()} {'='*20}")
    if not chunks:
        print("Brak segmentów do wyświetlenia.")
        return
    for idx in random.sample(range(len(chunks)), min(num_samples, len(chunks))):
        print(f"\n[Segment ID: {idx}]")
        print("-" * 50)
        print(chunks[idx])
        print("-" * 50)

preview_random_chunks(player_chunks, "Baza Gracza",  num_samples=1)
preview_random_chunks(master_chunks, "Baza Mistrza", num_samples=1)


==================== LOSOWY PODGLĄD: BAZA GRACZA ====================

[Segment ID: 163]
--------------------------------------------------
at 16th level.
After you use your breath weapon, you cant use it 
again until you complete a short or long rest.
Damage Resistance. You have resistance to the 
damage type associated with your draconic ancestry.
Languages. You can speak, read, and write Com m on 
and Draconic. Draconic is thought to be one of the 
oldest languages and is often used in the study of magic. 
The language sounds harsh to most other creatures and 
includes num erous hard consonants and sibilants.
--------------------------------------------------

==================== LOSOWY PODGLĄD: BAZA MISTRZA ====================

[Segment ID: 1515]
--------------------------------------------------
to observe creatures moving through the dungeo n 


DUNGEO N: LAIR 

Purpose 

d20 

1 
2 
3 

Armory stocked with weapons and armor 

Audience chamber, used to receive guests 

Banquet

In [ ]:
import faiss
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

def create_index(chunks):
    embeddings = model.encode(chunks, show_progress_bar=True)
    embeddings = np.array(embeddings).astype("float32")
    dimension = embeddings.shape[1]
    index = faiss.IndexFlatL2(dimension)
    index.add(embeddings)
    return index

player_index = create_index(player_chunks)
master_index = create_index(master_chunks)

print(f"Baza Player: {player_index.ntotal} wektorów")
print(f"Baza Master: {master_index.ntotal} wektorów")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/54 [00:00<?, ?it/s]

Batches:   0%|          | 0/100 [00:00<?, ?it/s]

Baza Player: 1709 wektorów
Baza Master: 3198 wektorów


In [ ]:
import numpy as np

def search_vector_db(query, index, chunks, k=2):
    query_vector = model.encode([query]).astype("float32")
    distances, indices = index.search(query_vector, k)
    print(f"\nZapytanie: {query}")
    for i in range(k):
        idx = indices[0][i]
        print(f"--- Wynik {i+1} (Dystans L2: {distances[0][i]:.4f}) ---")
        print(chunks[idx])
        print("-" * 50)

print("--- TEST BAZY MISTRZA ---")
search_vector_db("What is armor class of merfolk?", master_index, master_chunks)

print("\n--- TEST BAZY GRACZA ---")
search_vector_db("You cast alarm against what?", player_index, player_chunks)

--- TEST BAZY MISTRZA ---

Zapytanie: What is armor class of merfolk?
--- Wynik 1 (Dystans L2: 0.7472) ---
Merfolk defend their communities with spears 
crafted from whatever materials they can salvage from 
shipwrecks, beaches, and dead undersea creatures. 

1 MERFOLK 

Medium humanoid (merfolk), neutral 

Armor Class 11 
Hit Points 11 (2d8  2) 
Speed 10ft., swim 40ft. 

STR 
10 (0) 

CHA 
12 (1) 

DEX 
13 (1) 

CON 
12 (1) 

INT 
11 (0) 

WIS 
11 (0) 

Skills Perception 2 

Senses passive Perception 12 
Languages Aquan, Common 
Challenge 18 (25 XP) 

Amphibious. The merfolk can breathe air and water. 

ACTIONS 

Spear. Melee or Ranged Weapon Attack: 2 to hit, reach 5 ft. 
or range 2060 ft., one target. Hit: 3 (1 d6) piercing damage, 
or 4 (1d8) piercing damage if used with two hands to make a 
melee attack. 


MERROW 

MERROW 
Large monstrosity, chaotic evil 

Armor Class 13 (natural armor) 
Hit Points 45 (6d10  12) 
Speed 10ft., swim 40ft. 

STR 
18 (4) 
DEX 
10 (0) 

CON 
15 (2) 



In [ ]:
import json
import faiss

with open('player_chunks.json', 'w', encoding='utf-8') as f:
    json.dump(player_chunks, f, ensure_ascii=False, indent=4)
faiss.write_index(player_index, 'player.index')

with open('master_chunks.json', 'w', encoding='utf-8') as f:
    json.dump(master_chunks, f, ensure_ascii=False, indent=4)
faiss.write_index(master_index, 'master.index')

print("Zapisano obie bazy wektorowe (pliki .json oraz .index).")

Zapisano obie bazy wektorowe (pliki .json oraz .index).
